## Deep Learning y Sistemas Inteligentes - Laboratorio 8
- Josue Marroquin 22397
- Sebastian Huertas 22295

In [17]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

### 1) Preparación de datos

In [7]:
import pandas as pd

# Cargar datos
train = pd.read_csv("data/lab8/train.csv", parse_dates=["date"])
test = pd.read_csv("data/lab8/test.csv", parse_dates=["date"])

# Revisar duplicados
train.drop_duplicates(inplace=True)

# Revisar valores nulos
train = train.fillna(0)

# Features temporales
train["year"] = train["date"].dt.year
train["month"] = train["date"].dt.month
train["day"] = train["date"].dt.day
train["dayofweek"] = train["date"].dt.dayofweek
train["is_weekend"] = (train["dayofweek"] >= 5).astype(int)

# Features de lags
for lag in [1, 7, 14, 30]:
    train[f"lag_{lag}"] = train.groupby(["store","item"])["sales"].shift(lag)

# Rolling means
train["rolling_7"] = train.groupby(["store","item"])["sales"].shift(1).rolling(7).mean()
train["rolling_30"] = train.groupby(["store","item"])["sales"].shift(1).rolling(30).mean()


### 2) Preprocesamiento de datos

a) Division de Series temporales

In [10]:
df = df.sort_values("date")

# Rango de fechas
fecha_min = df["date"].min()
fecha_max = df["date"].max()

print("Rango disponible en el dataset:")
print("Fecha mínima:", fecha_min)
print("Fecha máxima:", fecha_max)

Rango disponible en el dataset:
Fecha mínima: 2013-01-01
Fecha máxima: 2017-12-31


In [11]:
train = df[(df["date"] >= "2013-01-01") & (df["date"] <= "2016-12-31")]
val   = df[(df["date"] >= "2017-01-01") & (df["date"] <= "2017-09-30")]
test  = df[(df["date"] >= "2017-10-01") & (df["date"] <= "2017-12-31")]

print("Tamaños:")
print("Train:", train.shape)
print("Val:", val.shape)
print("Test:", test.shape)

Tamaños:
Train: (730500, 4)
Val: (136500, 4)
Test: (46000, 4)


b) Generación de secuencias

In [15]:
def create_sequences(data, window_size=90, horizon=90):
    """
    data: serie de ventas (ordenada por fecha)
    window_size: días de historial
    horizon: días a predecir
    """
    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i : i + window_size])
        y.append(data[i + window_size : i + window_size + horizon])
    return np.array(X), np.array(y)

serie = train[(train["store"]==1) & (train["item"]==1)]["sales"].values
X_train, y_train = create_sequences(serie, window_size=90, horizon=90)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

def prepare_sequences(df, window_size=90, horizon=90):
    X_all, y_all = [], []
    for (store, item), group in df.groupby(["store", "item"]):
        serie = group.sort_values("date")["sales"].values
        X, y = create_sequences(serie, window_size, horizon)
        if len(X) > 0:  # evitar series demasiado cortas
            X_all.append(X)
            y_all.append(y)
    return np.vstack(X_all), np.vstack(y_all)

# Crear secuencias para train y val
X_train, y_train = prepare_sequences(train, window_size=90, horizon=90)
X_val, y_val     = prepare_sequences(val, window_size=90, horizon=90)

print("Dataset final:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)



X_train shape: (1282, 90)
y_train shape: (1282, 90)
Dataset final:
X_train: (641000, 90) y_train: (641000, 90)
X_val: (47000, 90) y_val: (47000, 90)


### 4) Seleccion de modelo
En este caso, lo mejor es LSTM, porque: Captura dependencias largas (ej. efectos anuales).Ha demostrado funcionar muy bien en pronóstico de ventas multiserie (como en el dataset de Kaggle). Aunque más lento que GRU, vale la pena por la riqueza del dataset

### 5) Entrenamiento del modelo

In [18]:
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_val   = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))

In [20]:
# ============================================================
# Modelo LSTM optimizado para entrenar rápido
# ============================================================
model = Sequential()

# LSTM más pequeño
model.add(LSTM(32, activation='tanh', return_sequences=False, input_shape=(X_train.shape[1], 1)))
model.add(Dropout(0.2))

# Capa densa intermedia reducida
model.add(Dense(16, activation='relu'))

# Capa de salida
model.add(Dense(y_train.shape[1], activation='linear'))

# Compilación
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# ============================================================
# Entrenamiento rápido
# ============================================================
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,          # SOLO 5 épocas
    batch_size=64,     # batch más grande = menos iteraciones
    verbose=1
)

# ============================================================
# Evaluación
# ============================================================
val_loss, val_mae = model.evaluate(X_val, y_val, verbose=0)
print(f" Validación -> Loss (MSE): {val_loss:.4f}, MAE: {val_mae:.4f}")


Epoch 1/5
10016/10016 ━━━━━━━━━━━━━━━━━━━━ 140s 14ms/step - loss: 257.8515 - mae: 11.3910 - val_loss: 208.0032 - val_mae: 10.7906
Epoch 2/5
10016/10016 ━━━━━━━━━━━━━━━━━━━━ 133s 13ms/step - loss: 187.0263 - mae: 10.2676 - val_loss: 202.9574 - val_mae: 10.6179
Epoch 3/5
10016/10016 ━━━━━━━━━━━━━━━━━━━━ 143s 14ms/step - loss: 169.7639 - mae: 9.7812 - val_loss: 202.1480 - val_mae: 10.5669
Epoch 4/5
10016/10016 ━━━━━━━━━━━━━━━━━━━━ 144s 14ms/step - loss: 159.2033 - mae: 9.4788 - val_loss: 162.3312 - val_mae: 9.5674
Epoch 5/5
10016/10016 ━━━━━━━━━━━━━━━━━━━━ 169s 17ms/step - loss: 141.6873 - mae: 8.9522 - val_loss: 151.5951 - val_mae: 9.2755
 Validación -> Loss (MSE): 151.5952, MAE: 9.2755


# Task 2

## ¿Cuál es el problema del gradiente de fuga en las redes LSTM y cómo afecta la efectividad de LSTM para el pronóstico de series temporales?

Aunque las LSTM fueron diseñadas específicamente para mitigar el problema del gradiente desvaneciente (vanishing gradient), aún pueden sufrir de "gradient leakage" o fuga de gradiente cuando las secuencias son extremadamente largas. Esto ocurre cuando la información se pierde gradualmente a través de las puertas (gates) durante la propagación hacia atrás, especialmente si las puertas de olvido están constantemente cerca de 0. En el pronóstico de series temporales, esto limita la capacidad del modelo para capturar dependencias a muy largo plazo, reduciendo su efectividad cuando los patrones importantes están separados por muchos pasos de tiempo.

## ¿Cómo se aborda la estacionalidad en los datos de series temporales cuando se utilizan LSTM para realizar pronósticos y qué papel juega la diferenciación en el proceso?

La estacionalidad en series temporales se puede abordar mediante varias estrategias: (1) incluir características temporales explícitas como mes, día de la semana o trimestre como variables de entrada adicionales, (2) aplicar diferenciación estacional para eliminar componentes periódicos antes de alimentar los datos al LSTM, o (3) usar arquitecturas híbridas que combinen descomposición estacional (como STL) con LSTM. La diferenciación estacional (restando valores de períodos anteriores) ayuda a hacer la serie estacionaria, lo que facilita el aprendizaje de patrones al LSTM al eliminar tendencias y ciclos recurrentes que podrían confundir al modelo.

## ¿Cuál es el concepto de "tamaño de ventana" en el pronóstico de series temporales con LSTM y cómo afecta la elección del tamaño de ventana a la capacidad del modelo para capturar patrones a corto y largo plazo?

El tamaño de ventana define cuántos pasos de tiempo históricos se utilizan como entrada para predecir el siguiente valor o secuencia futura. Un tamaño de ventana pequeño (ej. 5-10 pasos) permite al modelo capturar patrones a corto plazo y responder rápidamente a cambios recientes, pero puede perder dependencias de largo plazo. Un tamaño grande (ej. 50-100 pasos) captura mejor tendencias y patrones a largo plazo, pero aumenta la complejidad computacional y puede incluir información irrelevante o ruido. La elección óptima depende de la naturaleza de los datos: series con ciclos largos requieren ventanas mayores, mientras que series volátiles se benefician de ventanas más cortas.